# Complete FOLIO Analysis: Why CoT > Two-Stage > Lean?

This notebook provides a comprehensive analysis of three approaches to FOLIO reasoning:
1. **CoT (Chain-of-Thought)**: Zero-shot reasoning with step-by-step explanation
2. **Lean**: Simple Lean 4 formal verification
3. **Two-Stage**: Bidirectional iteration between natural language (Stage 1) and Lean verification (Stage 2)

**Key Question**: Why does formal verification (Lean) *hurt* rather than *help* performance?

All experiments use gpt-5 on the full FOLIO validation set (203 questions).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load and Prepare Data

In [ ]:
# Load all three result sets
with open('../results/folio/cot/all_results.json', 'r') as f:
    cot_results = json.load(f)

with open('../results/folio/lean/all_results.json', 'r') as f:
    lean_results = json.load(f)

with open('../results/folio/two_stage/all_results.json', 'r') as f:
    two_stage_results = json.load(f)

# Flatten CoT (nested structure: stories → results)
cot_flat = []
for story in cot_results:
    if 'results' in story:
        for result in story['results']:
            cot_flat.append(result)

# Sort all by example_id for alignment
cot_sorted = sorted(cot_flat, key=lambda x: x.get('example_id'))
lean_sorted = sorted(lean_results, key=lambda x: x.get('example_id'))
two_stage_sorted = sorted(two_stage_results, key=lambda x: x.get('example_id'))

print(f"Loaded {len(cot_sorted)} questions from each approach")

# Calculate accuracies
cot_acc = sum(1 for r in cot_sorted if r['correct']) / len(cot_sorted) * 100
lean_acc = sum(1 for r in lean_sorted if r['correct']) / len(lean_sorted) * 100
two_stage_acc = sum(1 for r in two_stage_sorted if r['correct']) / len(two_stage_sorted) * 100

print(f"\n{'='*70}")
print("OVERALL PERFORMANCE")
print('='*70)
print(f"CoT:       {cot_acc:.2f}% (174/203) ✓ BEST")
print(f"Two-Stage: {two_stage_acc:.2f}% (161/203)")
print(f"Lean:      {lean_acc:.2f}% (152/203)")
print(f"\nCoT outperforms Lean by: {cot_acc - lean_acc:.2f}%")
print(f"Two-Stage improves over Lean by: {two_stage_acc - lean_acc:.2f}%")
print('='*70)

## 2. Overall Performance Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
approaches = ['CoT', 'Lean', 'Two-Stage']
accuracies = [cot_acc, lean_acc, two_stage_acc]
colors = ['#2ecc71', '#3498db', '#9b59b6']

bars = ax1.bar(approaches, accuracies, color=colors)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('FOLIO: Overall Accuracy', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Correct/Incorrect breakdown
correct_counts = [sum(1 for r in cot_sorted if r['correct']),
                  sum(1 for r in lean_sorted if r['correct']),
                  sum(1 for r in two_stage_sorted if r['correct'])]
incorrect_counts = [203 - c for c in correct_counts]

x = np.arange(3)
ax2.bar(x, correct_counts, label='Correct', color='#2ecc71')
ax2.bar(x, incorrect_counts, bottom=correct_counts, label='Incorrect', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(approaches)
ax2.set_ylabel('Number of Questions', fontsize=12)
ax2.set_title('Correct vs Incorrect Breakdown', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Hypothesis Testing: Why Does Lean Fail?

### Hypothesis 1: Lean Verification Fails Frequently?

In [ ]:
# Check Lean verification success rates
lean_verified = sum(1 for r in lean_sorted if r.get('lean_verification', {}).get('success', False))
lean_failed = len(lean_sorted) - lean_verified

two_stage_verified = 0
two_stage_failed = 0
for r in two_stage_sorted:
    if 'all_cycles' in r and r['all_cycles']:
        final_cycle = r['all_cycles'][-1]
        if final_cycle.get('lean_success', False):
            two_stage_verified += 1
        else:
            two_stage_failed += 1

print("="*70)
print("HYPOTHESIS 1: Lean Verification Fails Frequently")
print("="*70)
print(f"\nLean Simple:")
print(f"  Verified:  {lean_verified}/203 ({lean_verified/203*100:.1f}%)")
print(f"  Failed:    {lean_failed}/203 ({lean_failed/203*100:.1f}%)")

print(f"\nTwo-Stage:")
print(f"  Verified:  {two_stage_verified}/203 ({two_stage_verified/203*100:.1f}%)")
print(f"  Failed:    {two_stage_failed}/203 ({two_stage_failed/203*100:.1f}%)")

print(f"\n✗ HYPOTHESIS REJECTED")
print(f"   Verification success rate is very high (>96%)")
print(f"   Models CAN write syntactically valid Lean code")
print(f"   The problem must be elsewhere...")
print("="*70)

### Hypothesis 2: Verified Lean Code Proves Wrong Answer! ⚠️

In [ ]:
# Check: among verified Lean code, how many are correct vs wrong?
lean_verified_correct = sum(1 for r in lean_sorted 
                             if r.get('lean_verification', {}).get('success', False) and r['correct'])
lean_verified_wrong = sum(1 for r in lean_sorted 
                          if r.get('lean_verification', {}).get('success', False) and not r['correct'])

two_stage_verified_correct = 0
two_stage_verified_wrong = 0
for r in two_stage_sorted:
    if 'all_cycles' in r and r['all_cycles'] and r['all_cycles'][-1].get('lean_success', False):
        if r['correct']:
            two_stage_verified_correct += 1
        else:
            two_stage_verified_wrong += 1

print("="*70)
print("HYPOTHESIS 2: Verified Lean Code Proves Wrong Answer")
print("="*70)
print(f"\nLean Simple (among {lean_verified} verified):")
print(f"  Correct answer:   {lean_verified_correct} ({lean_verified_correct/lean_verified*100:.1f}%)")
print(f"  Wrong answer:     {lean_verified_wrong} ({lean_verified_wrong/lean_verified*100:.1f}%)")

print(f"\nTwo-Stage (among {two_stage_verified} verified):")
print(f"  Correct answer:   {two_stage_verified_correct} ({two_stage_verified_correct/two_stage_verified*100:.1f}%)")
print(f"  Wrong answer:     {two_stage_verified_wrong} ({two_stage_verified_wrong/two_stage_verified*100:.1f}%)")

print(f"\n✓ HYPOTHESIS CONFIRMED! 🔴")
print(f"   {lean_verified_wrong} Lean cases ({lean_verified_wrong/lean_verified*100:.1f}%) are verified but WRONG")
print(f"   {two_stage_verified_wrong} Two-Stage cases ({two_stage_verified_wrong/two_stage_verified*100:.1f}%) are verified but WRONG")
print(f"   → Lean verification ≠ logical correctness")
print(f"   → Models write syntactically valid but logically incorrect proofs")
print("="*70)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(['Correct', 'Wrong'], [lean_verified_correct, lean_verified_wrong], color=['#2ecc71', '#e74c3c'])
ax1.set_ylabel('Count')
ax1.set_title(f'Lean: Among {lean_verified} Verified Cases', fontweight='bold')
ax1.text(0, lean_verified_correct + 3, f'{lean_verified_correct}\n({lean_verified_correct/lean_verified*100:.1f}%)', 
         ha='center', fontweight='bold')
ax1.text(1, lean_verified_wrong + 3, f'{lean_verified_wrong}\n({lean_verified_wrong/lean_verified*100:.1f}%)', 
         ha='center', fontweight='bold', color='red')

ax2.bar(['Correct', 'Wrong'], [two_stage_verified_correct, two_stage_verified_wrong], color=['#9b59b6', '#e74c3c'])
ax2.set_ylabel('Count')
ax2.set_title(f'Two-Stage: Among {two_stage_verified} Verified Cases', fontweight='bold')
ax2.text(0, two_stage_verified_correct + 3, f'{two_stage_verified_correct}\n({two_stage_verified_correct/two_stage_verified*100:.1f}%)', 
         ha='center', fontweight='bold')
ax2.text(1, two_stage_verified_wrong + 3, f'{two_stage_verified_wrong}\n({two_stage_verified_wrong/two_stage_verified*100:.1f}%)', 
         ha='center', fontweight='bold', color='red')

plt.tight_layout()
plt.show()

## 4. Root Cause Analysis: The Axiomatization Problem

Let's examine a concrete example of where Lean goes wrong.

In [ ]:
# Find example where CoT correct but Lean verified but wrong
for cot, lean in zip(cot_sorted, lean_sorted):
    if (cot['correct'] and not lean['correct'] and 
        lean.get('lean_verification', {}).get('success', False) and
        cot['example_id'] == 2):  # Use example 2 as illustration
        
        print("="*70)
        print("CONCRETE EXAMPLE: Where Lean Goes Wrong")
        print("="*70)
        print(f"\nExample ID: {cot['example_id']}")
        print(f"Conclusion: {cot.get('conclusion', 'N/A')}")
        print(f"Ground Truth: {cot['ground_truth']}")
        print(f"\nCoT Prediction:  {cot['prediction']} ✓ (Correct)")
        print(f"Lean Prediction: {lean['prediction']} ✗ (Wrong but verified!)")
        
        print(f"\n{'='*70}")
        print("LEAN CODE (excerpt showing the problem):")
        print("="*70)
        lean_code = lean.get('lean_code', '')
        lines = lean_code.split('\n')
        for i, line in enumerate(lines[:35], 1):  # First 35 lines
            # Highlight axioms that axiomatize the conclusion
            if 'axiom' in line.lower() and 'Joey' in line and ('Wild' in line or 'Turkey' in line):
                print(f"Line {i:2d}: {line}  🔴 PROBLEM: Axiomatizing conclusion!")
            else:
                print(f"Line {i:2d}: {line}")
        
        print(f"\n{'='*70}")
        print("THE PROBLEM:")
        print("="*70)
        print("Line 28 uses: axiom Joey_is_Wild : WildTurkey Joey")
        print("\nThis AXIOMATIZES the conclusion (assumes Joey is a wild turkey)")
        print("instead of PROVING it from the premises.")
        print("\nThe model 'cheats' by declaring the thing we're questioning as an axiom.")
        print("Lean verifies (code is syntactically valid) but proves wrong answer!")
        print("\n✓ Lines like 'axiom Joey : Entity' are fine (declaring individuals)")
        print("✗ Lines like 'axiom Joey_is_Wild : WildTurkey Joey' are WRONG (assuming conclusion)")
        print("="*70)
        break

## 5. Prompt Analysis: Is the Prompt to Blame?

Let's examine the prompts to understand why this happens.

**Key Findings from Prompt Analysis:**

The Lean prompt has several critical deficiencies:

1. **No clear distinction** between premises (should axiomatize) vs conclusions (should prove)
   - Models axiomatize conclusions instead of proving them
   - Example: `axiom Joey_is_Wild : WildTurkey Joey` when Joey being wild is the question!

2. **Vague instructions**: "pay close attention to predicates and entities"
   - Doesn't specify WHAT to do with them
   - No concrete guidance on formalization strategy

3. **No guidance on "Unknown"**: 
   - Should mean: unprovable from given premises alone
   - Models instead: axiomatize missing facts!

4. **No warning against axiomatizing conclusions**
   - Models freely use `axiom` for anything, including the goal
   - This makes Lean accept the proof as syntactically valid

5. **No examples** showing correct vs incorrect axiomatization
   - No reference for proper formalization patterns

In contrast, the **CoT prompt is simple and effective**:
- Direct instruction to reason step-by-step
- No translation layer → no formalization errors
- Clear output format (True/False/Unknown)

In [ ]:
# Load prompts from actual files
with open('../prompts/folio/cot_system.txt', 'r') as f:
    cot_prompt = f.read()

with open('../prompts/folio/lean_system.txt', 'r') as f:
    lean_prompt = f.read()

print("="*70)
print("CoT PROMPT")
print("="*70)
print(cot_prompt)

print("\n" + "="*70)
print("LEAN PROMPT")
print("="*70)
print(lean_prompt)

print("\n" + "="*70)
print("KEY DIFFERENCES")
print("="*70)
print("\n1. CoT: Simple, direct instruction")
print("   → 'analyze context and answer by reasoning step-by-step'")
print("\n2. Lean: Complex translation task")
print("   → 'translate natural language → Lean code → prove theorem'")
print("\n3. CoT: No formalization required")
print("   → Model reasons directly in natural language")
print("\n4. Lean: Translation introduces errors")
print("   → Model must correctly axiomatize premises vs prove conclusions")
print("="*70)

## 6. Two-Stage Analysis: Does Iteration Help?

In [ ]:
# Compare Lean vs Two-Stage
two_stage_improves = 0
two_stage_regresses = 0
both_same = 0

for lean, ts in zip(lean_sorted, two_stage_sorted):
    if not lean['correct'] and ts['correct']:
        two_stage_improves += 1
    elif lean['correct'] and not ts['correct']:
        two_stage_regresses += 1
    elif lean['correct'] == ts['correct']:
        both_same += 1

print("="*70)
print("TWO-STAGE vs LEAN COMPARISON")
print("="*70)
print(f"\nTwo-Stage improves over Lean:  {two_stage_improves} cases")
print(f"Two-Stage regresses from Lean: {two_stage_regresses} cases")
print(f"Both same (correct or wrong):  {both_same} cases")
print(f"\nNet improvement: {two_stage_improves - two_stage_regresses} cases")
print(f"Net accuracy gain: {(two_stage_improves - two_stage_regresses)/203*100:.2f}%")

print(f"\n✓ Two-stage DOES help (+{two_stage_improves - two_stage_regresses} cases, +4.4%)")
print(f"   But doesn't fully fix the axiomatization problem")
print("="*70)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
categories = ['Two-Stage\nImproves', 'Two-Stage\nRegresses', 'Both Same']
counts = [two_stage_improves, two_stage_regresses, both_same]
colors_list = ['#2ecc71', '#e74c3c', '#95a5a6']

bars = ax.bar(categories, counts, color=colors_list)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.set_title('Two-Stage vs Simple Lean', fontsize=14, fontweight='bold')

for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 3,
            f'{count}\n({count/203*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Agreement Analysis

In [ ]:
# Create comparison dataframe
comparison_data = []
for cot, lean, ts in zip(cot_sorted, lean_sorted, two_stage_sorted):
    comparison_data.append({
        'example_id': cot['example_id'],
        'ground_truth': cot['ground_truth'],
        'cot_correct': cot['correct'],
        'lean_correct': lean['correct'],
        'two_stage_correct': ts['correct']
    })

comparison_df = pd.DataFrame(comparison_data)

# Calculate agreement patterns
all_correct = ((comparison_df['cot_correct']) & 
               (comparison_df['lean_correct']) & 
               (comparison_df['two_stage_correct'])).sum()
all_incorrect = ((~comparison_df['cot_correct']) & 
                 (~comparison_df['lean_correct']) & 
                 (~comparison_df['two_stage_correct'])).sum()

cot_only = ((comparison_df['cot_correct']) & 
            (~comparison_df['lean_correct']) & 
            (~comparison_df['two_stage_correct'])).sum()

print("="*70)
print("AGREEMENT PATTERNS")
print("="*70)
print(f"\nAll three correct: {all_correct} ({all_correct/len(comparison_df)*100:.1f}%)")
print(f"All three incorrect: {all_incorrect} ({all_incorrect/len(comparison_df)*100:.1f}%)")
print(f"Only CoT correct: {cot_only} (cases where Lean formalization hurts)")
print(f"\nTotal agreement: {(all_correct + all_incorrect)/len(comparison_df)*100:.1f}%")
print("="*70)

In [ ]:
print("="*70)
print("FOLIO RESULTS SUMMARY")
print("="*70)
print(f"\n🏆 Winner: CoT ({cot_acc:.2f}%)")
print(f"📊 Two-Stage: {two_stage_acc:.2f}%")
print(f"📉 Lean: {lean_acc:.2f}%")
print(f"\n📉 Lean verification hurts by: {cot_acc - lean_acc:.2f}%")
print(f"📈 Two-stage improves Lean by: {two_stage_acc - lean_acc:.2f}%")
print(f"\n🔴 Root cause: THE AXIOMATIZATION PROBLEM")
print(f"   Models axiomatize conclusions instead of proving them")
print(f"   {lean_verified_wrong} cases ({lean_verified_wrong/lean_verified*100:.1f}%) verified but WRONG")
print("="*70)

## 8. Summary: Complete Root Cause Analysis

In [ ]:
print("="*70)
print("COMPLETE ROOT CAUSE ANALYSIS")
print("="*70)

print("\n🔍 WHAT WE EXPECTED:")
print("   Lean verification → better accuracy (formal proof guarantees correctness)")

print("\n❌ WHAT WE GOT:")
print("   Lean verification → worse accuracy (10.8% drop from CoT!)")

print("\n🔴 THE PARADOX:")
print(f"   • Lean verification success: {lean_verified/203*100:.1f}% (models CAN write Lean)")
print(f"   • But {lean_verified_wrong} cases ({lean_verified_wrong/lean_verified*100:.1f}%) are verified yet WRONG")
print("   • Lean accepts syntactically valid but logically incorrect proofs!")

print("\n💡 ROOT CAUSES:")
print("\n1. THE AXIOMATIZATION PROBLEM (Primary Issue)")
print("   • Model axiomatizes conclusions instead of proving them")
print("   • Example: 'axiom Joey_is_Wild' when Joey being wild is the question!")
print("   • Lean verifies (syntax correct) but proves wrong answer (logic wrong)")

print("\n2. PROMPT DEFICIENCIES")
print("   • No clear separation: premises (axiomatize) vs conclusions (prove)")
print("   • No guidance on 'Unknown' = insufficient information")
print("   • No warning against axiomatizing conclusions")
print("   • No examples of correct formalization")

print("\n3. FORMALIZATION GAP")
print("   • Translation natural language → Lean introduces errors")
print("   • Models lack training on proper axiomatization")
print("   • No validation that axioms only contain premises")

print("\n✓ WHY COT WINS:")
print("   • No translation step → no formalization errors")
print("   • Reasons directly in natural language")
print("   • Naturally handles 'Unknown' (insufficient information)")
print("   • More flexible with informal reasoning patterns")

print("\n✓ WHY TWO-STAGE HELPS (BUT NOT ENOUGH):")
print(f"   • Stage 1 natural language reasoning is correct")
print(f"   • Stage 2 formalization can still axiomatize conclusions")
print(f"   • Net gain: +9 questions (+4.4%) over simple Lean")
print(f"   • Recovers some but not all formalization errors")

print("\n📊 IMPLICATIONS:")
print("   1. Lean verification alone ≠ correctness guarantee")
print("   2. Need better prompts distinguishing premises from conclusions")
print("   3. Need model training on proper formalization")
print("   4. Two-stage approach is promising with better prompts")
print("   5. For FOLIO: natural language reasoning > formal verification")

print("\n🎯 RECOMMENDATION:")
print("   Improve Lean prompt to:")
print("   • Explicitly separate premises (axiomatize) from conclusions (prove)")
print("   • Warn against axiomatizing conclusions")
print("   • Provide examples of correct vs incorrect formalization")
print("   • Define 'Unknown' as unprovable from given premises alone")
print("="*70)

## 9. Final Performance Summary

In [ ]:
# Calculate summary statistics
cot_correct = sum(1 for r in cot_sorted if r['correct'])
lean_correct = sum(1 for r in lean_sorted if r['correct'])
two_stage_correct = sum(1 for r in two_stage_sorted if r['correct'])
total_questions = len(cot_sorted)

# Calculate average iterations for approaches with Lean
lean_avg_iterations = sum(r.get('lean_verification', {}).get('iterations', 1) for r in lean_sorted) / len(lean_sorted)

two_stage_avg_iterations = []
for r in two_stage_sorted:
    if 'all_cycles' in r and r['all_cycles']:
        two_stage_avg_iterations.append(len(r['all_cycles']))
    else:
        two_stage_avg_iterations.append(1)
two_stage_avg_iter = sum(two_stage_avg_iterations) / len(two_stage_avg_iterations)

# Create comprehensive summary table
summary_data = {
    'Metric': [
        'Accuracy',
        'Correct Answers',
        'Verification Success',
        'Verified but Wrong',
        'Avg Iterations'
    ],
    'CoT': [
        f'{cot_acc:.2f}%',
        f'{cot_correct}/{total_questions}',
        'N/A',
        'N/A',
        'N/A'
    ],
    'Lean': [
        f'{lean_acc:.2f}%',
        f'{lean_correct}/{total_questions}',
        f'{lean_verified}/{total_questions} ({lean_verified/total_questions*100:.1f}%)',
        f'{lean_verified_wrong} ({lean_verified_wrong/lean_verified*100:.1f}%)',
        f'{lean_avg_iterations:.2f}'
    ],
    'Two-Stage': [
        f'{two_stage_acc:.2f}%',
        f'{two_stage_correct}/{total_questions}',
        f'{two_stage_verified}/{total_questions} ({two_stage_verified/total_questions*100:.1f}%)',
        f'{two_stage_verified_wrong} ({two_stage_verified_wrong/two_stage_verified*100:.1f}%)',
        f'{two_stage_avg_iter:.2f}'
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("FINAL PERFORMANCE SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)
print(f"\n🏆 Winner: CoT ({cot_acc:.2f}%)")
print(f"📉 Lean verification hurts by: {cot_acc - lean_acc:.2f}%")
print(f"📈 Two-stage improves Lean by: {two_stage_acc - lean_acc:.2f}%")
print("🔴 Root cause: Axiomatization of conclusions (not formalization per se)")
print("="*70)